Digital Business University of Applied Sciences

Data Science und Management (M. Sc.)

DAMI01 / DATA01 Data Analytics

Prof. Dr. Daniel Ambach

Julia Schmid (200022)

***
# Time-Series-Clustering
# Teil 2: Data Preparation
***


In [128]:
# Imports
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
import webbrowser
import os

from parameter import *
from funktionen import * 

In [ ]:
# Einstellung, sodass alle Spalten eines Datensatzes angezeigt werden
pd.set_option('display.max_columns', None) 

In [100]:
# Daten laden
path_temp = INPUT_PATH + "/data_acquisition.csv"
df = pd.read_csv(path_temp)

***
## Überblick


In [ ]:
# Anzahl der Zeilen und Spalten ausgeben
print(f'Anzahl Zeilen: {df.shape[0]}')
print(f'Anzahl Spalten: {df.shape[1]}')

In [ ]:
# Zehn zufällige Einträge ausgeben
df.sample(n=10)

In [ ]:
# Datensatz-Info ausgeben
df.info()

***
## Phase 3: Datenvorbereitung
***
### **Datenqualität**

### Duplikate

In [ ]:
# Anzahl der Duplikate bestimmen
df_duplicates = df[df.duplicated()]
print(f'Dieser Datensatz besitz {len(df_duplicates)} Duplikate.')

### Fehlende Werte

In [ ]:
# Fehlende Werte ermitteln
count_nan = df.isna().sum()

# Prozentsatz der NaN-Werte berechnen
percent_nan = round((count_nan / len(df)) * 100, 2)

# Werte in einem DataFrame speichern
df_nan = pd.DataFrame({
    'Anzahl NaN': count_nan,
    'Prozent NaN': percent_nan
}).sort_values(by='Anzahl NaN', ascending=False)

# Variablen mit fehlenden Werten inkl. Anzahl und Prozentanteil ausgeben
df_nan[df_nan['Anzahl NaN']>0]

Fehlende Werte treten ausschließlich bei kategorischen Variablen auf. Diese werden daher durch die Kategorien *Unknown* oder *None* ersetzt.

Bei der Variable **errors** lassen sich fehlende Werte inhaltlich als „kein Fehler aufgetreten" deuten. Sie werden daher mit dem Wert *None* gefüllt.

Bei den Variablen **zip** und **merchant_state** gibt es hingegen keinen inhaltlichen Anhaltspunkt, welcher konkrete Wert hinter den fehlenden Einträgen stecken könnte. Diese werden daher mit *Unknown* gekennzeichnet.

In [106]:
# Fehlende Werte der Variable "errors" als "None" kennzeichnen
df["errors"] = df["errors"].fillna("None")

# Variable "zip" als String speichern und fehlende Werte als "Unknown" kennzeichnen
df["zip"] = pd.to_numeric(df["zip"], errors="coerce").astype("Int64").astype("string")
df["zip"] = df["zip"].fillna("Unknown")

# Fehlende Werte der Variable "merchant_state" als "Unknown" kennzeichnen
df["merchant_state"] = df["merchant_state"].fillna("Unknown")

### Vollständigkeit

In [ ]:
gesamt_zellen = df.shape[0] * df.shape[1]
fehlende_zellen = df.isna().sum().sum()
vollstaendigkeit = ((gesamt_zellen - fehlende_zellen) / gesamt_zellen) * 100
print(f"Datenvollständigkeit: {vollstaendigkeit:.2f}%")

# Quelle: Ambach, D. data analytics master. Abgerufen am 10.03.2026 von https://github.com/dan-am/data analytics master/blob/main/3 data preparation/data validation.py

### **Irrelevante Spalten löschen**

Eine Begründung, warum diese Spalten gelöscht werden, steht im Abschnitt *Teil B: Data Understanding* in der Datei *01_ Business_Data_Understanding.ipynb*.

In [108]:
df = df.drop(
    columns=[
        "merchant_id",
        "id_cards",
        "id_user",
        "card_number",
        "expires",
        "cvv",
        "card_on_dark_web",
        "mcc",
        "card_type",
        "has_chip",
        "acct_open_date",
        "address",
        "latitude",
        "longitude",
        "current_age",
        "retirement_age",
        "merchant_city",
        "use_chip",
    ]
)

### **Kategorische Variablen**

In [ ]:
# Kategorische Variablen bestimmen
categoricalVar = [col for col in df if df[col].dtype == 'object']

# Eindeutige Werte der kategorischen Variablen ausgeben
for i in categoricalVar:
    print(i)
    print(df[i].unique()) # Eindeutige Werte bestimmen
    print('')

Die Variable **date** wird in das Datenformat Datum geändert und Tag ohne die Uhrzeitangabe (yyy-mm-dd) als separate Variable betrachtet.

Bei den Variablen **amount**, **credit_limit**,  **per_capita_income**,  **yearly_income** und **total_debt** wird das Dollarzeichen entfernt und als numerische Variable gespeichert.

Für die Variablen **description**, **errors**,  **merchant_state** werden Kategorien gebildet, um die Werte zu gruppieren und die Auswertung zu vereinfachen.


In [110]:
# Variable "date" als Datetime speichern
df['date'] = pd.to_datetime(df['date'])

# Tag aus dem Variable "date" extrahieren
df['day'] = df['date'].dt.date

In [111]:
# Dollarzeichen entfernen und als numerische Spalten speichern
list_dollar_change = [
    "amount",
    "credit_limit",
    "per_capita_income",
    "yearly_income",
    "total_debt",
]

for i in list_dollar_change:
    df[i] = df[i].str[1:] # Dollarzeichen entfernen
    df[i] = pd.to_numeric(df[i], errors="coerce") # In numerische Spalte umwandeln

In [ ]:
# Kategorien für "description" erstellen
df['description_category'] = df['description'].apply(lambda x: get_key_value(x, map_description))
df = df.drop(columns=["description"])
df["description_category"].value_counts()

In [ ]:
# Kategorien für "merchant_state" erstellen
df['merchant_category'] = df['merchant_state'].apply(lambda x: get_key_value(x, map_merchant_state))
df = df.drop(columns=["merchant_state"])
df['merchant_category'].value_counts()

In [ ]:
# Kategorien für "errors" erstellen
df['error_category'] = df['errors'].apply(lambda x: get_key_value(x, map_error_category))
df = df.drop(columns=["errors"])
df["error_category"].value_counts()

### **Numerischen Variablen**

In [ ]:
# Numerische und kategorische Variablen ausgeben
numericalVar = [col for col in df if df[col].dtype != 'object']
for i in range(0, len(numericalVar), 10):
    print(", ".join(numericalVar[i:i+10]))

Die Variable **zip**, **birth_year** , **birth_month**, wird als Kategorische Variable gespeichert.

In [116]:
list_numeric_change = ["zip", "birth_year", "birth_month"]
for i in list_numeric_change:
    df[i] = df[i].astype(str) # Umwandlung in String
    df[i].value_counts() # Anzahl der Werte anzeigen

### **Cleaned-Datensatz**

In [ ]:
print("Neuer Cleaned-Datensatz:", df.shape)
display(df.head())
df.info()

***
## Phase 4: Feature Engineering
***
Aus dem aufbereiteten Datensatz werden zwei Teildatensätze erstellt: eine Zeitreihe mit den Variablen card_id, day und amount, die als Grundlage für das Clustering dient, sowie ein ergänzender Datensatz mit den übrigen Variablen. Der Datensatz mit den ergänzenden Informationen wird im Anschluss herangezogen, um die gebildeten Cluster inhaltlich zu analysieren und die Ergebnisse zu bewerten.


### **Zeitreihen**

Für alle Karten-ID wird für jeden Tag ein Eintrag erstellt (vollständige Zeitreihe)


In [ ]:
# Datensatz bei dem pro Karten-ID die Summe der Umsätze (amount) pro Tag (day) steht
df_timeseries_pre = (
    df.groupby(["card_id", "day"])
    .agg(amount_sum=("amount", "sum"))
    .reset_index()
)

# Tag in ein Datetime-Format bringen
df_timeseries_pre["day"] = pd.to_datetime(df_timeseries_pre["day"])

# Liste aller Karten-IDs
list_cards = df_timeseries_pre["card_id"].unique() 
# Liste aller Tage zwischen den Min-Tag und Max-Tag die im Datensatz vorkommen
list_days = pd.date_range(
    df_timeseries_pre["day"].min(),
    df_timeseries_pre["day"].max(),
)

# Vollständige Karte-Tag-Kombinationen erstellen
combination_card_day = pd.MultiIndex.from_product(
    [list_cards, list_days],
    names=["card_id", "day"]
)

# Erstellen eines Datensatz, bei dem jede Karte für jeden Tag den tatsächlichen Umsatz oder den Wert 0 hat
df_timeseries = (
    df_timeseries_pre.set_index(["card_id", "day"])
      .reindex(combination_card_day, fill_value=0)
      .reset_index()
)

df_timeseries.head()

Die Karten-IDs, bei denen mehr als 50 Prozent der Tageswerte den Betrag 0 aufweisen, werden aus dem Datensatz entfernt, da sie zu wenig Transaktionsaktivität zeigen und somit nur eine geringe Aussagekraft für das Clustering besitzen.

In [ ]:
# Anteil Nullwerte pro Karte
cards_null_percent = df_timeseries.groupby("card_id")["amount_sum"].apply(lambda x: 100 * (x ==0).mean())

# Karten weniger 50% 
cards_over_50 = cards_null_percent[cards_null_percent > 70].index

# Karten die mehr als 50% Null-Werte haben, werdne nicht weiter betrachtet
df_timeseries= df_timeseries[df_timeseries["card_id"].isin(cards_over_50)]


In [121]:
# Liste der Karten-IDs
list_card_ids = df_timeseries['card_id'].unique()

In [122]:
# Pivotisierung: jede Zeile repräsentiert eine Karte
# Fehlende Werte werden mit 0 gefüllt
df_timeseries = df_timeseries.pivot(
    index='card_id', columns='day', values='amount_sum'
).fillna(0)

In [123]:
# Daten speichern
path_temp = INPUT_PATH + "/data_timeseries.csv"
df_timeseries.to_csv(path_temp, index=True)

### **Zusatzinfos**
Ergänzend wird ein Zusatzinfo-Datensatz erstellt, der die übrigen Variablen pro Karte zusammenfasst. Dabei werden numerische Spalten durch ihren Median und kategoriale Spalten durch den jeweils häufigsten Wert aggregiert. Die Ergebnisse werden anschließend zu einem Datensatz zusammengeführt, sodass zu jede card_id eindeutig zusammengefassten Informationen vorliegen.

In [124]:
df_add_info = df.drop(["date", "id_transaction", "amount", "day"], axis=1)

# Nur Karten-IDs, die in der Zeitreihe sind
df_add_info = df_add_info[df_add_info["card_id"].isin(list_card_ids)]

In [125]:
# Numerische Spalten aggregieren (Median)
numericalVar = df_add_info.select_dtypes(include='number').columns
df_num = df_add_info.groupby("card_id")[numericalVar].median()

# Kategorische Spalten aggregieren (häufigster Wert)
categoricalVar = df_add_info.select_dtypes(exclude='number').columns
df_cat = df_add_info.groupby("card_id")[categoricalVar].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)

# Numerische und kategorische Ergebnisse zusammenführen
df_zusatzinfo = df_num.join(df_cat)

# card_id Spalte löschen, da diese Spalte als Index gespeichert ist
df_zusatzinfo = df_zusatzinfo.drop("card_id", axis=1)

# df_zusatzinfo.head()

In [126]:
# Daten speichern
path_temp = INPUT_PATH + "/data_add_info.csv"
df_zusatzinfo.to_csv(path_temp, index=True)

In [ ]:
# Profilingreport der Zusatzinfos laden 
pr = ProfileReport(df_zusatzinfo, title = 'Credit Data') 
filename_pr = "../output/add_info_data_pr.html" 
path_pr = os.path.abspath(filename_pr) 

pr.to_file(path_pr)  # ProfileReport als HTML speichern
webbrowser.open(f"file://{path_pr}")  # ProfileReport im Browser öffnen

***

In [97]:
# # Alle Variablen und Objekte aus dem Arbeitsspeicher löschen / Speicher freigeben 
%reset -f

***
***